# Simple Iris Classification Model using Tensorflow

## Data Preparation

### Import Library

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np
import os
import requests

### Load Dataset

In [13]:
iris = load_iris()
X = iris.data
y = iris.target

In [16]:
class_names = iris.target_names
print(class_names)

['setosa' 'versicolor' 'virginica']


## Data Preprocessing

### One Hot Encoding

In [19]:
y_encoded = to_categorical(y)

### Data Split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

## Modelling

### Build and Compile Model

In [21]:
# Normalisasi data
normalizer = tf.keras.layers.Normalization(input_shape=(4,))
normalizer.adapt(X_train)

model = Sequential([
    normalizer,
    Dense(10, activation='relu',),
    Dense(8, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

c:\Users\HP\miniconda3\envs\iris-classification\lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Training

In [22]:
model.fit(X_train, y_train, epochs=50, batch_size=5, verbose=1, validation_split=0.1)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2593 - loss: 1.1608 - val_accuracy: 0.4167 - val_loss: 1.0371
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4074 - loss: 1.0049 - val_accuracy: 0.5000 - val_loss: 0.9625
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6389 - loss: 0.8844 - val_accuracy: 0.5833 - val_loss: 0.9024
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6759 - loss: 0.7968 - val_accuracy: 0.5833 - val_loss: 0.8565
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6759 - loss: 0.7344 - val_accuracy: 0.5833 - val_loss: 0.8172
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6667 - loss: 0.6835 - val_accuracy: 0.5833 - val_loss: 0.7826
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6759 - loss: 0.6393 - val_accuracy: 0.5833 - val_loss: 0.7535
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6759 - loss: 0.6042 - val_accuracy: 0.5833 - val_loss

### Evaluation

In [23]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.9667


### Export Model & Scaler

In [32]:

path = "models/iris/1"
os.makedirs(path, exist_ok=True)
tf.saved_model.save(model, path)

INFO:tensorflow:Assets written to: models/iris/1\assets


INFO:tensorflow:Assets written to: models/iris/1\assets


## Result

### Predict with New Data

In [20]:
# Load model
model = tf.saved_model.load("models/iris/1")

new_input = np.array([[6.0,3.4,4.5,1.6,]])

infer = model.signatures["serving_default"]

input_tensor = tf.convert_to_tensor(new_input, dtype=tf.float32)

prediction = infer(input_tensor)

prediction_values = list(prediction.values())[0].numpy()

predicted_class = np.argmax(prediction_values, axis=1)

print("Prediction result:", class_names[predicted_class[0]])


Prediction result: versicolor


In [ ]:
endpoint = "http://localhost:8501/v1/models/iris:predict"

data = {
    "instances": [
        [6.2, 3.4, 5.4, 2.3]
    ]
}

response = requests.post(endpoint, json=data)

result = response.json()

pred_class = np.argmax(result["predictions"][0])

print("Predicted class:", class_names[pred_class])

Predicted class: virginica
